In [4]:
%load_ext autoreload
%autoreload 2

import sys
import os

from notebooks.jacob.round3.dna_build_parser import DNABuildParser, PCRComponents, Aliquot, create_pcr_plate_assignments, create_echo_instructions

# Initialize the parser
parser = DNABuildParser('notebooks/jacob/round3/')
print("DNABuildParser initialized successfully!")

# Test getBuildNames
build_names = parser.getBuildNames()
print(f"Available builds ({len(build_names)}):")
for build in build_names:
    print(f"  - {build}")

# Test getAssemblyNamesForBuild for first build
test_build = build_names[0]
assembly_names = parser.getAssemblyNamesForBuild(test_build)
print(f"Assembly products for {test_build} ({len(assembly_names)}):")
for assembly in assembly_names[:5]:  # Show first 5
    print(f"  - {assembly}")
if len(assembly_names) > 5:
    print(f"  ... and {len(assembly_names) - 5} more")

# Test getAssemblyPCRs for first assembly
test_assembly = assembly_names[0]
pcr_names = parser.getAssemblyPCRs(test_assembly)
print(f"PCRs for {test_assembly} ({len(pcr_names)}):")
for pcr in pcr_names:
    print(f"  - {pcr}")

# Test getPCRComponents for first PCR
test_pcr = pcr_names[0]
primer_vol = 2.5  # μL
template_vol = 1.0  # μL

components = parser.getPCRComponents(test_pcr, primer_vol, template_vol)
print(f"Components for {test_pcr}:")
print(f"  Left Primer:  {components.left_primer}")
print(f"  Right Primer: {components.right_primer}")
print(f"  Template:     {components.template}")

# Test all components
all_pcr_components_found = True
for build in  parser.getBuildNames():
    for assembly in parser.getAssemblyNamesForBuild(build):
        for pcr in parser.getAssemblyPCRs(assembly):
            try:
                components = parser.getPCRComponents(pcr, primer_vol, template_vol)
            except Exception as e:
                print(f"Error getting components for {pcr}: {e}")
                all_pcr_components_found = False
assert all_pcr_components_found, "Some PCR components were not found!"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 249 oligos from IDT specsheet
Loaded 0 R2_random oligos from secondary CSV
DNABuildParser initialized successfully!
Available builds (6):
  - AP_OplR_CH_R3
  - AP_OplR_DNL_R3
  - AP_OplR_EVL_R3
  - AP_OplR_R2_random
  - GAH_DBAT_R3
  - LK_BorAT_R3
Assembly products for AP_OplR_CH_R3 (24):
  - pAP_OplR_CH_R3_0001
  - pAP_OplR_CH_R3_0002
  - pAP_OplR_CH_R3_0003
  - pAP_OplR_CH_R3_0004
  - pAP_OplR_CH_R3_0005
  ... and 19 more
PCRs for pAP_OplR_CH_R3_0001 (2):
  - PCR_AP_OplR_CH_R3_0001
  - PCR_AP_OplR_CH_R3_0002
Components for PCR_AP_OplR_CH_R3_0001:
  Left Primer:  Aliquot(source_plate='r3_oligos_combined', source_well='C19', volume=2.5, expected_sequence='CGACCACCAAGCGAAACATCGCATCGAGCGAGCACGTACTCGG')
  Right Primer: Aliquot(source_plate='r3_oligos_combined', source_well='C10', volume=2.5, expected_sequence='CTGGCGCTCTGGCTGTGGCCAGC')
  Template:     Aliquot(source_plate='template_plate', sourc

In [4]:
build_pcr_blocks = {
  'AP_OplR_CH_R3': ('r3_pcrs_p1', 'A1', 'C1'),
  'AP_OplR_DNL_R3': ('r3_pcrs_p1', 'E1', 'G1'),
  'AP_OplR_EVL_R3': ('r3_pcrs_p2', 'A1', 'C1'),
  'AP_OplR_R2_random': ('r3_pcrs_p2', 'E1', 'G1'),
  'GAH_DBAT_R3': ('r3_pcrs_p3', 'A1', 'C1'),
  'LK_BorAT_R3': ('r3_pcrs_p4', 'A1', 'C1'),
}
assignments = create_pcr_plate_assignments(build_pcr_blocks, parser)
# assignments

In [12]:
echo_df = create_echo_instructions(assignments, parser, primer_volume=60, template_volume=60)
echo_df.to_csv('notebooks/jacob/round3/251001_echo_instructions_dbat_oplr.csv', index=False)

# REDO SOME OplR BUILDS

I've minipreped proper version of some of the OplR templates that were bad and led to bad builds:
* pAP_OplR_R2_0021 (Q123D_L241I)
* pAP_OplR_R2_0025 (Q123D_Q296H)
* pAP_OplR_R2_0026 (Q123D_Q303R)
* pAP_OplR_R2_0039 (Y146A_Y255T)

I am going to redo DNL and EVL

In [5]:
oplr_redo_pcr_blocks = {
  'AP_OplR_DNL_R3': ('r3_pcrs_p1', 'A1', 'C1'),
  'AP_OplR_EVL_R3': ('r3_pcrs_p2', 'E1', 'G1'),
}
oplr_redo_assignments = create_pcr_plate_assignments(oplr_redo_pcr_blocks, parser)
oplr_redo_echo_df = create_echo_instructions(oplr_redo_assignments, parser, primer_volume=60, template_volume=60)
oplr_redo_echo_df.to_csv('notebooks/jacob/round3/251015_echo_instructions_oplr_redo.csv', index=False)